# LC 42 — Trapping Rain Water
**Difficulty:** Hard | **Category:** Monotonic Stack / Two Pointers
**Pattern:** Two-Pointer with Running Max

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Water at any position is limited by
the shorter of its left max and right max walls, minus its own height.
Two pointers let us process whichever side has the smaller max,
because that side is the binding constraint — no need to look further.
</div>

## Official Problem Statement

Given `n` non-negative integers representing an elevation map where
the width of each bar is `1`, compute how much water it can trap
after raining.

**Constraints:**
- `n == height.length`
- `1 <= n <= 20_000`
- `0 <= height[i] <= 100_000`

## What This Is Actually Asking

Imagine vertical walls of different heights on a flat surface.
When it rains, water fills the valleys between taller walls.
For each column, water trapped = min(tallest wall left, tallest wall
right) minus the column's own height (floored at 0).
Sum those values across all columns.
Return the total units of trapped water.

## Walk Through an Example by Hand

```
height = [0,1,0,2,1,0,1,3,2,1,2,1]
indices   [0,1,2,3,4,5,6,7,8,9,A,B]

Two pointers: left=0, right=11, left_max=0, right_max=0, water=0

left_max=0 < right_max=0 (equal → left branch):
  h[0]=0 >= left_max=0 → left_max=0, water+=0-0=0.  left=1
  h[1]=1 >= left_max=0 → left_max=1, water+=1-1=0.  left=2
  h[2]=0 <  left_max=1 → water+=1-0=1.  left=3
  h[3]=2 >= left_max=1 → left_max=2, water+=0.  left=4

left_max=2 > right_max=0 → right branch:
  h[11]=1 >= right_max=0 → right_max=1. right=10
  h[10]=2 >= right_max=1 → right_max=2. right=9
  h[9]=1  <  right_max=2 → water+=2-1=1. right=8
  h[8]=2  >= right_max=2 → right_max=2. right=7

left=4, right=7, left_max=2, right_max=2:
  left branch: h[4]=1<2 → water+=1. left=5
  left branch: h[5]=0<2 → water+=2. left=6
  left branch: h[6]=1<2 → water+=1. left=7
  left==right → stop

Total water = 0+1+0+1+1+2+1 = 6
```

## The Picture

```
height = [0,1,0,2,1,0,1,3,2,1,2,1]

  3 |                        ██
  2 |         ██       ██ ██    ██
  1 |      ██    ██ ██ ██ ██ ██ ██ ██
  0 |  ██  ██  ██  ██ ██ ██ ██ ██ ██ ██ ██ ██
       [0] [1] [2] [3] [4] [5] [6] [7] [8] [9][10][11]

Water fills the gaps (~~ = water):
  3 |                        ██
  2 |         ██ ~~  ~~  ~~ ██ ██ ~~ ██
  1 |      ██ ~~ ██  ██~~ ██ ██ ██ ██ ██

Two-pointer logic:
  Move the pointer on the SMALLER max side.
  That side is the binding constraint for water level.
  The other side's wall is guaranteed tall enough.

  left_max < right_max → water bounded by left_max → move left
  left_max >= right_max → water bounded by right_max → move right
```

## When To Use This Pattern

- When water (or any fill) at a position depends on constraints from
  both sides, think two pointers with running max.
- When the bottleneck is always the smaller of two running maxima,
  think: move the pointer on the smaller side.
- When a precomputed prefix/suffix max array would work but you want
  O(1) space, think two pointers.
- When values at each position are bounded by a min of two global
  extremes, think two pointers converging inward.
- When the problem has symmetric left/right structure, think two
  pointers from both ends.

## The Approach

Set left and right pointers at the two ends, and track left_max and
right_max.
At each step, compare left_max and right_max: move the pointer on
the smaller-max side, because that side determines how much water
can be held — the other side is guaranteed to be at least as tall.
Add (current_max - height[pointer]) to total water, then advance
the pointer.
Stop when left meets right.

In [2]:
from typing import List  # type hints for function signatures

In [3]:
def test_harness(func):
    """Run test cases for trap."""
    tests = [
        # (height, expected_water)
        ([0,1,0,2,1,0,1,3,2,1,2,1], 6),
        ([4,2,0,3,2,5],              9),
        ([1,0,1],                    1),
        ([3,0,2,0,4],                7),
        ([0,0,0],                    0),
        ([1],                        0),
    ]
    passed = 0
    for i, (height, expected) in enumerate(tests):
        result = func(height)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(f"Test {i+1}: {status}")
        if status == "FAILED":
            print(f"  Input:    {height}")
            print(f"  Expected: {expected}")
            print(f"  Got:      {result}")
    print(f"\n{passed}/{len(tests)} tests passed.")

In [7]:
def trap(height: List[int]) -> int:
    """
    Calculate total water trapped between elevation bars.

    Args:
        height: list of non-negative bar heights
    Returns:
        total units of trapped rain water

    Approach:
        Two pointers (left, right) + running left_max, right_max.
        Move the pointer on the smaller-max side.
        water += current_max - height[pointer] at each step.
    """
    if not height: return 0
    l, r = 0, len(height) - 1
    leftMax, rightMax = height[l], height[r]
    res = 0
    while l < r:
        if leftMax <= rightMax:
            l += 1
            leftMax = max(leftMax, height[l])
            res += leftMax-height[l]

        else:
            r -= 1 
            rightMax = max(rightMax, height[r])
            res += rightMax - height[r]
    return res
    
"""
6
9
1
7
Test 1: PASSED
Test 2: PASSED
Test 3: PASSED
Test 4: PASSED
Test 5: PASSED
Test 6: PASSED

6/6 tests passed.
"""

# --- Debug prints (expected in comments) ---
h1 = [0,1,0,2,1,0,1,3,2,1,2,1]
print(trap(h1))         # 6

h2 = [4,2,0,3,2,5]
print(trap(h2))         # 9

h3 = [1,0,1]
print(trap(h3))         # 1

h4 = [3,0,2,0,4]
print(trap(h4))         # 7
test_harness(trap)

6
9
1
7
Test 1: PASSED
Test 2: PASSED
Test 3: PASSED
Test 4: PASSED
Test 5: PASSED
Test 6: PASSED

6/6 tests passed.


In [ ]:
# Uncomment and run when solution is ready
# test_harness(trap)

## Complexity

| Approach              | Time   | Space  |
|-----------------------|--------|--------|
| Brute Force           | O(n²)  | O(1)   |
| Prefix/Suffix Arrays  | O(n)   | O(n)   |
| Two Pointers          | O(n)   | O(1)   |
| Monotonic Stack       | O(n)   | O(n)   |

Two pointers is optimal: single pass, constant extra space.

## Real World Connection

At Citi, latency spikes across 6,000 endpoints form a profile similar
to an elevation map — some services spike high, others stay low.
"Trapped water" maps to the total latency budget absorbed by
intermediate services sandwiched between high-latency outliers.
Identifying these pockets helps the DE team pinpoint which services
inflate end-to-end pipeline latency on AWS Step Functions.
The two-pointer O(n) scan processes a full hour of time-bucketed
telemetry without materializing two extra arrays.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra